In [1]:
import pandas as pd
import numpy as np
from elasticsearch import Elasticsearch

client = Elasticsearch("http://localhost:9200")
client.ping()


True

In [2]:
csv_file = "datos_merge_completo.csv"
df = pd.read_csv(csv_file)

df.shape


/tmp/ipykernel_4768/2648448857.py:2: DtypeWarning: Columns (28,35,37,39,40,41,44,46) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


(51707, 49)

In [3]:
mapa_provincias = {
    "CORDOBA": "ES-CO",
    "MALAGA": "ES-MA",
    "MADRID": "ES-M",
    "BARCELONA": "ES-B",
    "BIZKAIA": "ES-BI",
    "GIPUZKOA": "ES-SS",
    "ARABA": "ES-VI",
    "ALAVA": "ES-VI",
    "NAVARRA": "ES-NA",
    "VALENCIA": "ES-V"
}

df["codigo_region"] = df["City"].astype(str).str.upper().map(mapa_provincias)

df["codigo_region"].isna().sum()


np.int64(43984)

In [4]:
mapa_iso = {
    "CORDOBA": "ES-CO",
    "MALAGA": "ES-MA",
    "MADRID": "ES-M",
    "BARCELONA": "ES-B",
    "VALENCIA": "ES-V",
    "SEVILLA": "ES-SE",
    "BIZKAIA": "ES-BI",
    "GIPUZKOA": "ES-SS",
    "ARABA": "ES-VI"
}

df["codigo_region"] = df["provincia"].str.upper().map(mapa_iso)


In [5]:
# Fechas
date_cols = ["booked_at", "checkin_time"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Dinero
num_cols = ["reservation_net_value", "total_adr"]
for col in num_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "")
            .astype(float)
        )

# Booleanos
bool_cols = ["all_entry_forms_completed", "returning_inhabitant"]
mapper = {"yes": True, "no": False, True: True, False: False}
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map(mapper)

# Nulos
df = df.replace({np.nan: None, pd.NaT: None})


In [6]:
if client.indices.exists(index="kibana_definitivo"):
    client.indices.delete(index="kibana_definitivo")
    print("🗑️ Índice kibana_definitivo eliminado")


🗑️ Índice kibana_definitivo eliminado


In [7]:
mapping = {
    "mappings": {
        "properties": {
            "codigo_region": {"type": "keyword"},
            "codigo_ccaa": {"type": "keyword"},
            "City": {"type": "keyword"},
            "status": {"type": "keyword"},
            "business_segment": {"type": "keyword"},
            "reservation_net_value": {"type": "double"},
            "checkin_time": {
                "type": "date",
                "format": "yyyy-MM-dd HH:mm:ss"
            }
        }
    }
}

client.indices.create(
    index="kibana_definitivo",
    body=mapping
)


/tmp/ipykernel_4768/3257564030.py:18: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  client.indices.create(


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'kibana_definitivo'})

In [8]:
print(df.columns.tolist())


['checkin_time', 'lead_time', 'lenght_of_stay', 'checkin_month', 'checkin_day', 'adult_count', 'child_count', 'origin', 'requested_category', 'asset', 'asset_type', 'business_segment', 'rate_type', 'returning_inhabitant', 'recurrence', 'libere_community', 'reservation_net_value', 'zona_roja_mes', 'Estacion_Estancia', 'Estacion_Reserva', 'estancia_en_festivo', 'estancia_en_finde', 'City', 'mes_checkin_numerico', 'reservation_checkin_month_diff', 'Reserva_Dia_Semana', 'status', 'fecha', 'indicativo', 'nombre', 'provincia', 'altitud', 'tmed', 'prec', 'tmin', 'horatmin', 'tmax', 'horatmax', 'dir', 'velmedia', 'racha', 'horaracha', 'hrMedia', 'hrMax', 'horaHrMax', 'hrMin', 'horaHrMin', 'ciudad', 'codigo_ccaa', 'codigo_region']


In [9]:
contador = 0

for _, row in df.iterrows():
    doc = row.to_dict()
    
    # Formato fechas según mapping
    for col in date_cols:
        if col in doc and doc[col] is not None:
            # Usar espacio entre fecha y hora
            doc[col] = doc[col].strftime("%Y-%m-%d %H:%M:%S")
    
    client.index(index="kibana_definitivo", document=doc)
    contador += 1

    if contador % 1000 == 0:
        print(f"✅ {contador} filas subidas")

print(f"🏁 FIN — Total filas indexadas: {contador}")


✅ 1000 filas subidas
✅ 2000 filas subidas
✅ 3000 filas subidas
✅ 4000 filas subidas
✅ 5000 filas subidas
✅ 6000 filas subidas
✅ 7000 filas subidas
✅ 8000 filas subidas
✅ 9000 filas subidas
✅ 10000 filas subidas
✅ 11000 filas subidas
✅ 12000 filas subidas
✅ 13000 filas subidas
✅ 14000 filas subidas
✅ 15000 filas subidas
✅ 16000 filas subidas
✅ 17000 filas subidas
✅ 18000 filas subidas
✅ 19000 filas subidas
✅ 20000 filas subidas
✅ 21000 filas subidas
✅ 22000 filas subidas
✅ 23000 filas subidas
✅ 24000 filas subidas
✅ 25000 filas subidas
✅ 26000 filas subidas
✅ 27000 filas subidas
✅ 28000 filas subidas
✅ 29000 filas subidas
✅ 30000 filas subidas
✅ 31000 filas subidas
✅ 32000 filas subidas
✅ 33000 filas subidas
✅ 34000 filas subidas
✅ 35000 filas subidas
✅ 36000 filas subidas
✅ 37000 filas subidas
✅ 38000 filas subidas
✅ 39000 filas subidas
✅ 40000 filas subidas
✅ 41000 filas subidas
✅ 42000 filas subidas
✅ 43000 filas subidas
✅ 44000 filas subidas
✅ 45000 filas subidas
✅ 46000 filas subid

In [ ]:
df.columns

Index(['checkin_time', 'lead_time', 'lenght_of_stay', 'checkin_month',
       'checkin_day', 'adult_count', 'child_count', 'origin',
       'requested_category', 'asset', 'asset_type', 'business_segment',
       'rate_type', 'returning_inhabitant', 'recurrence', 'libere_community',
       'reservation_net_value', 'zona_roja_mes', 'Estacion_Estancia',
       'Estacion_Reserva', 'estancia_en_festivo', 'estancia_en_finde', 'City',
       'mes_checkin_numerico', 'reservation_checkin_month_diff',
       'Reserva_Dia_Semana', 'status', 'fecha', 'indicativo', 'nombre',
       'provincia', 'altitud', 'tmed', 'prec', 'tmin', 'horatmin', 'tmax',
       'horatmax', 'dir', 'velmedia', 'racha', 'horaracha', 'hrMedia', 'hrMax',
       'horaHrMax', 'hrMin', 'horaHrMin', 'ciudad', 'codigo_ccaa',
       'codigo_region'],
      dtype='object')